# PHYT1D — Module 06: ODE Simulator

Integrates the augmented ODE system using Euler steps (Δt = 5 min).

**Features:**
- Exercise activation logic (in-bout / post-bout)
- Bolus computation (CHO/CR × bolus_factor)
- 1000-realisation uncertainty propagation
- Returns median + 25th/75th percentile bands


In [ ]:
import numpy as np


## 6.1 Meal and Insulin Rate Functions

In [ ]:
def build_CHO_rate(user_params, dt=1.0, T=1440):
    """
    Build a piecewise-constant CHO ingestion rate array [mg/kg/min].
    Assumes 75 kg adult (scale CHO g → mg/kg using body weight).

    Meals are spread uniformly over meal_duration [min] (default 20 min),
    matching the UVa/Padova trapezoidal ingestion profile (Dalla Man 2006).

    Parameters
    ----------
    user_params : dict — Group C parameters
    dt          : float — time step [min]
    T           : int   — total simulation length [min]

    Returns
    -------
    CHO_rate : np.ndarray shape (T//dt + 1,) — CHO rate at each time step
    """
    BW             = 75.0   # kg — population body weight
    meal_duration  = user_params.get("meal_duration", 20.0)  # min — eating window
    n_steps        = int(T / dt) + 1
    CHO_rate       = np.zeros(n_steps)
    n_meal_steps   = max(1, int(round(meal_duration / dt)))   # steps to spread over

    f = user_params.get("CHO_factor", 1.0)
    meals = {
        "BF": (user_params["t_BF"], user_params["CHO_BF"] * f),
        "LU": (user_params["t_LU"], user_params["CHO_LU"] * f),
        "DN": (user_params["t_DN"], user_params["CHO_DN"] * f),
        "SN": (user_params["t_SN"], user_params["CHO_SN"] * f),
    }
    for label, (t_meal, cho_g) in meals.items():
        idx_start = int(round(t_meal / dt))
        # Spread CHO uniformly over n_meal_steps (g → mg/kg/min)
        rate_per_step = (cho_g * 1000.0) / BW / (n_meal_steps * dt)
        for k in range(n_meal_steps):
            idx = idx_start + k
            if 0 <= idx < n_steps:
                CHO_rate[idx] += rate_per_step

    return CHO_rate


def build_bolus_rate(user_params, dt=1.0, T=1440):
    """
    Build a piecewise-constant insulin bolus rate array [mU/kg/min].

    Bolus = (CHO * CHO_factor / CR) * bolus_factor  [Units: U]
    Converted: 1 U = 1000 mU; 75 kg body weight.

    Bolus is delivered over bolus_duration [min] (default 2 min — rapid pump delivery),
    preventing a single-step insulin spike that causes an unphysiological glucose drop.

    Parameters
    ----------
    user_params : dict — Group C parameters

    Returns
    -------
    bolus_rate : np.ndarray — bolus rate at each step [mU/kg/min]
    """
    BW             = 75.0
    CR             = user_params["CR"]
    f              = user_params.get("CHO_factor", 1.0)
    bf             = user_params.get("bolus_factor", 1.0)
    bolus_duration = user_params.get("bolus_duration", 2.0)  # min — pump delivery window
    n_steps        = int(T / dt) + 1
    n_bolus_steps  = max(1, int(round(bolus_duration / dt)))  # steps to spread bolus over
    bolus_rate     = np.zeros(n_steps)

    meals = {
        "BF": (user_params["t_BF"], user_params["CHO_BF"] * f),
        "LU": (user_params["t_LU"], user_params["CHO_LU"] * f),
        "DN": (user_params["t_DN"], user_params["CHO_DN"] * f),
        "SN": (user_params["t_SN"], user_params["CHO_SN"] * f),
    }
    for label, (t_meal, cho_g) in meals.items():
        idx_start        = int(round(t_meal / dt))
        bolus_U          = (cho_g / CR) * bf
        # Spread bolus evenly over n_bolus_steps
        rate_per_step    = (bolus_U * 1000.0) / BW / (n_bolus_steps * dt)
        for k in range(n_bolus_steps):
            idx = idx_start + k
            if 0 <= idx < n_steps:
                bolus_rate[idx] += rate_per_step

    return bolus_rate


## 6.2 Single ODE Trajectory (Euler, Δt = 5 min)

In [ ]:
def simulate_trajectory(theta_A, theta_B, user_params, fixed, dt=1.0, T=1440):
    """
    Simulate a single 24-hour glucose trajectory.

    Parameters
    ----------
    theta_A     : dict — Group A params (SI, SG, Gb, p2, kd, ka2, kempt, kabs)
    theta_B     : dict — Group B params (beta_aer, beta_res, tau_post, tau_on, VO2max, phi)
    user_params : dict — Group C params (exercise prescription, meals, bolus)
    fixed       : dict — Group D fixed params
    dt          : float — Euler step [min]
    T           : int   — simulation duration [min]

    Returns
    -------
    times : np.ndarray [min]
    IG    : np.ndarray [mg/dL] — interstitial glucose (CGM-equivalent)
    G     : np.ndarray [mg/dL] — plasma glucose
    """
    # ─── Import ODE functions inline for portability ──────────────────────────
    def rho_fn(G, Gb):
        r1, r2, Gth = fixed["r1"], fixed["r2"], fixed["Gth"]
        if G >= Gb:     return 0.0
        elif G > Gth:   return (10**r1)*(np.log(max(G,1))**r2 - np.log(Gb)**r2)**2
        else:           return (10**r1)*(np.log(Gth)**r2 - np.log(Gb)**r2)**2

    # ─── Unpack parameters ───────────────────────────────────────────────────
    SI    = theta_A["SI"];    SG    = theta_A["SG"];   Gb    = theta_A["Gb"]
    p2    = theta_A["p2"];    kd    = theta_A["kd"];   ka2   = theta_A["ka2"]
    kempt = theta_A["kempt"]; kabs  = theta_A["kabs"]

    beta_aer = theta_B["beta_aer"]; beta_res = theta_B["beta_res"]
    tau_post = theta_B["tau_post"]; tau_on   = theta_B["tau_on"]
    phi_val  = theta_B.get("phi", 0.0)

    VI    = fixed["VI"];     ke    = fixed["ke"];    beta  = fixed["beta"]
    f     = fixed["f"];      VG    = fixed["VG"];    alpha = fixed["alpha"]
    Fc01  = fixed["Fc01"];   gamma_aer = fixed["gamma_aer"]
    gamma_res = fixed["gamma_res"]; tau_ramp = fixed["tau_ramp"]
    delta_res_val = fixed["delta_res"]

    u            = user_params["u"]
    d            = user_params["d"]
    t_start      = user_params["t_start"]
    ex_type      = user_params["exercise_type"]
    t_end_ex     = t_start + d

    # ─── Build input arrays ───────────────────────────────────────────────────
    n_steps = int(T / dt) + 1
    times   = np.arange(n_steps) * dt
    CHO_arr = build_CHO_rate(user_params, dt, T)
    INS_arr = build_bolus_rate(user_params, dt, T)

    # ─── Basal insulin (constant background rate) ─────────────────────────────
    Ipb        = 10.0                        # mU/L — physiological basal plasma insulin
    Isc2_b     = Ipb * ke / (ka2 + 1e-9)   # mU/kg
    Isc1_b     = Ipb * ke / (kd  + 1e-9)   # mU/kg
    basal_rate = Ipb * ke * fixed["VI"]      # mU/kg/min — true steady-state basal rate

    # ─── Initial state (basal steady state) ──────────────────────────────────
    state = {
        "Isc1" : Isc1_b,
        "Isc2" : Isc2_b,
        "Ip"   : Ipb,
        "Qsto1": 0.0,
        "Qsto2": 0.0,
        "Qgut" : 0.0,
        "G"    : Gb,
        "X"    : 0.0,
        "IG"   : Gb,
    }

    G_arr  = np.zeros(n_steps)
    IG_arr = np.zeros(n_steps)
    G_arr[0]  = state["G"]
    IG_arr[0] = state["IG"]

    # ─── Euler integration ────────────────────────────────────────────────────
    for i in range(n_steps - 1):
        t = times[i]

        # ── Exercise state ──
        if u > 0 and t_start <= t <= t_end_ex:
            t_ex    = t - t_start
            t_prime = 0.0
            active  = True
        elif u > 0 and t > t_end_ex:
            t_ex    = d
            t_prime = t - t_end_ex
            active  = False
        else:
            t_ex = 0.0; t_prime = 0.0; active = False

        # ── ΔSI ──
        if u > 0 and active:
            if ex_type == "aerobic":
                dSI_ex = beta_aer * u * (1.0 - np.exp(-t_ex / tau_on))
            else:
                dSI_ex = -delta_res_val * u * min(t_ex / tau_ramp, 1.0)
        else:
            dSI_ex = 0.0

        dSI_tail = beta_aer * u * np.exp(-t_prime / tau_post) if u > 0 and t_prime > 0 else 0.0
        SI_e     = SI * (1.0 + dSI_ex + dSI_tail)

        # ── ΔFc01 ──
        if u > 0 and active:
            ramp   = min(t_ex / tau_ramp, 1.0)
            g_coef = gamma_aer if ex_type == "aerobic" else gamma_res
            Fc01_e = Fc01 + g_coef * u * ramp
        else:
            Fc01_e = Fc01

        # ── ΔEGP (resistance only) ──
        if u > 0 and active and ex_type == "resistance":
            ramp    = min(t_ex / tau_ramp, 1.0)
            dEGP    = theta_B["beta_res"] * u**2 * ramp
        else:
            dEGP = 0.0

        # ── kempt_eff ──
        ke_emp = kempt * (1.0 + phi_val * u) if (u > 0 and active) else kempt

        # ── Insulin input (delayed by beta minutes) ──
        delay_idx = max(0, i - int(beta / dt))
        I_in  = INS_arr[delay_idx] + basal_rate   # bolus + basal

        # ── ODE updates (Euler) ──
        Isc1  = state["Isc1"]; Isc2 = state["Isc2"]; Ip = state["Ip"]
        Qsto1 = state["Qsto1"]; Qsto2 = state["Qsto2"]; Qgut = state["Qgut"]
        G     = state["G"]; X = state["X"]; IG = state["IG"]

        dIsc1  = -kd * Isc1 + I_in / VI
        dIsc2  = kd * Isc1 - ka2 * Isc2
        dIp_v  = ka2 * Isc2 - ke * Ip

        dQsto1 = -ke_emp * Qsto1 + CHO_arr[i]
        dQsto2 = ke_emp * Qsto1  - ke_emp * Qsto2
        dQgut  = ke_emp * Qsto2  - kabs * Qgut

        Ra_val = f * kabs * Qgut
        # rho(G) is for risk scoring only — not an ODE gate (Dalla Man 2014)
        dG_v   = -(SG + X) * G + SG * Gb + Ra_val / VG + dEGP
        dX_v   = -p2 * (X - SI_e * (Ip - Ipb))
        dIG_v  = -(1.0 / alpha) * (IG - G)

        state["Isc1"]  = max(Isc1  + dt * dIsc1,  0.0)
        state["Isc2"]  = max(Isc2  + dt * dIsc2,  0.0)
        state["Ip"]    = max(Ip    + dt * dIp_v,   0.0)
        state["Qsto1"] = max(Qsto1 + dt * dQsto1,  0.0)
        state["Qsto2"] = max(Qsto2 + dt * dQsto2,  0.0)
        state["Qgut"]  = max(Qgut  + dt * dQgut,   0.0)
        state["G"]     = max(G     + dt * dG_v,    20.0)
        state["X"]     = X + dt * dX_v
        state["IG"]    = max(IG    + dt * dIG_v,   20.0)

        G_arr[i+1]  = state["G"]
        IG_arr[i+1] = state["IG"]

    return times, IG_arr, G_arr


## 6.3 Uncertainty Propagation — 1000 Posterior Realisations

In [ ]:
def simulate_ensemble(realisations_A, realisations_B, user_params, fixed,
                      dt=1.0, T=1440, n_real=1000):
    """
    Run 1000 posterior realisations and return median + CI bands.

    Parameters
    ----------
    realisations_A : np.ndarray (n_real, 8)  — Group A posterior samples
    realisations_B : np.ndarray (n_real, 6)  — Group B posterior samples
    user_params    : dict — Group C
    fixed          : dict — Group D
    n_real         : int  — number of realisations to run

    Returns
    -------
    times  : np.ndarray [min]
    IG_50  : np.ndarray [mg/dL] — median
    IG_25  : np.ndarray [mg/dL] — 25th percentile
    IG_75  : np.ndarray [mg/dL] — 75th percentile
    all_IG : np.ndarray (n_real, n_steps)
    """
    param_names_A = ["SI","SG","Gb","p2","kd","ka2","kempt","kabs"]
    param_names_B = ["beta_aer","beta_res","tau_post","tau_on","VO2max","phi"]

    n_real_actual = min(n_real, len(realisations_A), len(realisations_B))
    idx = np.arange(n_real_actual)

    n_steps = int(T / dt) + 1
    all_IG  = np.zeros((n_real_actual, n_steps))

    for r in idx:
        theta_A = {k: realisations_A[r, j] for j, k in enumerate(param_names_A)}
        theta_B = {k: realisations_B[r, j] for j, k in enumerate(param_names_B)}
        try:
            times, ig, _ = simulate_trajectory(theta_A, theta_B, user_params, fixed, dt, T)
            all_IG[r] = ig
        except Exception as e:
            all_IG[r] = np.nan

    # Remove failed runs
    valid   = ~np.isnan(all_IG).any(axis=1)
    all_IG  = all_IG[valid]

    IG_50 = np.percentile(all_IG, 50, axis=0)
    IG_25 = np.percentile(all_IG, 25, axis=0)
    IG_75 = np.percentile(all_IG, 75, axis=0)

    return times, IG_50, IG_25, IG_75, all_IG


## 6.4 Quick Demo — Single Trajectory

In [ ]:
# Standalone demo with default parameter values
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Fixed params ──────────────────────────────────────────────────────────────
FIXED = {
    "VI":0.126,"ke":0.127,"beta":8.0,"f":0.90,"VG":1.45,"alpha":7.0,
    "r1":1.44,"r2":0.81,"Gth":60.0,"Fc01":0.0097,
    "gamma_aer":0.003,"gamma_res":0.0015,"tau_ramp":20.0,"delta_res":0.003,
}

# ── Default theta_A / theta_B (prior means) ───────────────────────────────────
theta_A_demo = {"SI":np.exp(-5.3),"SG":0.025,"Gb":120.0,"p2":0.05,
                "kd":0.026,"ka2":0.066,"kempt":0.055,"kabs":0.057}
theta_B_demo = {"beta_aer":0.008,"beta_res":0.005,"tau_post":240.0,
                "tau_on":15.0,"VO2max":45.2,"phi":0.1}

user_params_demo = {
    "u":0.50,"d":60.0,"t_start":420.0,"exercise_type":"aerobic",
    "CR":15.0,"CHO_BF":45.0,"CHO_LU":70.0,"CHO_DN":60.0,"CHO_SN":20.0,
    "t_BF":60.0,"t_LU":300.0,"t_DN":720.0,"t_SN":960.0,
    "bolus_factor":1.0,"CHO_factor":1.0,"sex":"M","age_group":"adult",
}

times, IG_demo, G_demo = simulate_trajectory(theta_A_demo, theta_B_demo,
                                              user_params_demo, FIXED)
clock = (times + 7*60) % (24*60) / 60  # hours from midnight

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times/60, IG_demo, color="steelblue", lw=2, label="IG (simulated)")
ax.axhline(70,  color="green",  ls="--", lw=1, alpha=0.7)
ax.axhline(180, color="orange", ls="--", lw=1, alpha=0.7)
ax.axvspan(user_params_demo["t_start"]/60,
           (user_params_demo["t_start"]+user_params_demo["d"])/60,
           alpha=0.15, color="red", label="Exercise bout")
ax.set_xlabel("Time [h from simulation start]", fontsize=11)
ax.set_ylabel("IG [mg/dL]", fontsize=11)
ax.set_title("Demo: Single Trajectory — 50% VO₂max, 60-min Aerobic", fontsize=12)
ax.legend(); plt.tight_layout(); plt.savefig("demo_trajectory.png", dpi=150)
plt.show()
print("✓ Simulator module ready.")
